In [ ]:
import sys, os
from pathlib import Path
_root = Path(os.getcwd())
while not ((_root / 'src').exists() and (_root / 'pyproject.toml').exists()) and _root != _root.parent:
    _root = _root.parent
sys.path.insert(0, str(_root))

# Fatos de manchetes por LLM + MiniLM (somente Colab GPU)

Extração local **no Colab** com Qwen2.5-7B-Instruct quantizado em 4 bits, sem API externa e sem corpos de artigos. Somente os títulos verificados de `fusion_events.jsonl.gz` entram no prompt. A saída estruturada descreve a notícia e o papel da emissora, não recomenda investimento nem prevê retorno. A extração é retomável pelo cache auditável no MyDrive; execute novamente a célula de extração após qualquer interrupção. **A célula de treino deve ser executada apenas após a extração completa.**

O pré-treino de Qwen2.5 pode conhecer resultados de 2020–2023: títulos isolados e citações não eliminam vazamento da memória paramétrica. O teste 2022-09 a 2023-11 já foi inspecionado anteriormente, logo o experimento é diagnóstico, não confirmação nova. Não há pressuposto de superioridade sobre previsão nula ou comprar e manter. Nenhum treino ou inferência é permitido no computador local.

In [ ]:
import importlib.util
import subprocess

try:
    from google.colab import drive
except ImportError as exc:
    raise RuntimeError('Execute exclusivamente no Google Colab com GPU') from exc
drive.mount('/content/drive')
REPO = Path('/content/latent_fusion')
if REPO.exists():
    if not (REPO / '.git').is_dir():
        raise RuntimeError(f'Pasta existente não é repositório Git: {REPO}')
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', 'https://github.com/rbfcdog/latent_fusion.git', str(REPO)], check=True)
if importlib.util.find_spec('torch') is None:
    raise RuntimeError('PyTorch indisponível; selecione GPU e reinicie')
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Ative GPU: Ambiente de execução → Alterar tipo de ambiente → T4/L4 GPU')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers>=4.44,<5', 'accelerate', 'bitsandbytes', 'sentence-transformers==2.3.1'], check=True)
DATA = Path('/content/drive/MyDrive/latent_fusion_data')
PANEL, NEWS = DATA / 'fusion_event_panel.npz', DATA / 'fusion_events.jsonl.gz'
for file in (PANEL, NEWS):
    if not file.is_file():
        raise FileNotFoundError(f'Envie o artefato auditado para {file}')
EMBEDDING_CACHE = DATA / 'article_embedding_cache'
LLM_CACHE = DATA / 'llm_headline_cache'
OUT = DATA / 'llm_attention_results'
sys.path.insert(0, str(REPO))
{'gpu': torch.cuda.get_device_name(0), 'painel': str(PANEL), 'manchetes': str(NEWS), 'cache_llm': str(LLM_CACHE), 'saída': str(OUT)}

## Extração completa e auditável

A célula seguinte infere cada manchete no Colab, mostra o progresso e registra resposta original, estrutura validada e evidência na saída JSONL do Drive. Se a sessão terminar antes do fim, reexecute esta célula: registros completos já gravados são aproveitados, sem recriar resultados anteriores. `status=invalid` indica falha de formato/proveniência, distinta de campos desconhecidos (`0`) em saídas válidas. Mais de 5% de saídas inválidas interrompem o experimento antes do treino.

In [ ]:
import gc
import json
import pandas as pd
from IPython.display import display
from src.fusion.llm_headlines import extract_llm_features
from src.fusion.llm_attention_experiment import _audit

facts, facts_path = extract_llm_features(NEWS, LLM_CACHE, batch_size=4)
audit = _audit(facts['features'], facts['status'])
display(pd.DataFrame([audit]))
with open(facts_path, encoding='utf-8') as stream:
    sample = [json.loads(next(stream)) for _ in range(min(3, audit['total']))]
display(pd.DataFrame([{key: row.get(key) for key in ('index', 'ticker', 'text', 'status', 'parsed', 'raw', 'error')} for row in sample]))
gc.collect()
torch.cuda.empty_cache()
if audit['invalid_pct'] > 5:
    raise ValueError(f"Extrações inválidas: {audit['invalid_pct']:.2f}% > 5%; examine o cache antes do treino")
{'cache_auditável': str(facts_path), 'manchetes_concluídas': audit['total']}

## Treino e avaliação cronológica

O ajuste começa somente após a extração completa e a liberação da GPU. A ablação MiniLM recebe os mesmos oito títulos em até cinco pregões, a mesma novidade calculada antes das notícias do mesmo dia, idade, máscaras, arquitetura e splits purgados; somente os campos estruturados pelo LLM são retirados. Preços usam o mesmo painel. Sem manchete recente, os dois modelos de atenção preveem excesso zero.

São comparados MSE geral e nas linhas com manchete, IC, retorno líquido com custos de 10 bps por nocional negociado, alpha/beta contra comprar e manter e previsão nula de excesso zero. Carteiras compartilham amostras e datas. Cinco pregões de alvo se sobrepõem; universo ex post inclui ETFs; ausência de manchete não prova ausência de notícia; janeiro de 2024 não tem cobertura auditada. A pasta nova `llm_attention_results/` contém métricas, previsões, curvas, históricos, checkpoints e relatório real após a execução.

In [ ]:
from IPython.display import Image
from src.fusion.llm_attention_experiment import run_experiment

result = run_experiment(PANEL, NEWS, OUT, EMBEDDING_CACHE, LLM_CACHE, cost_bps=10, llm_result=(facts, facts_path))
display(result['signal'], result['portfolio'], result['split_coverage'])
display(pd.DataFrame([result['audit']]), result['histories']['price'], result['histories']['llm'], result['histories']['embedding'])
display(Image(filename=str(OUT / 'llm_vs_controls.png')))
{'relatório': str(OUT / 'results.md'), 'previsões': str(OUT / 'predictions_llm.csv'), 'cache': str(result['llm_cache'])}